# Обзор

git - популярный инструмент для совместной работы с версионируемым кодом (SVC). Начинал разработывать Линус Торвальдс в 2005 году. По расшифровке есть разные версии, по одной из которых это дословно "мерзавец" в честь Торвальдса

Получил популярность благодарая своей простотеё. Проект развивается, добавляются новые фичи, многие из них связанны с производительностью. Из важного:<br>
- 2015 git worktree для нескольких рабочих каталогов
- 2019 git switch / git restore для более прозрачного переключения между ветками

#### Альтернативы
- до появления git в 2005 был популярен VSS от Microsoft<br>он по инерции еще долго использовался в больших консервативных бизнесах
- Apache Subversion (SVN) - открытая версия системы контроля версий
- Mercurial появился параллельно с git, рассматривается как его упрощенная версия

<img src="img/svn_tools.avif" width=750>

Сравнение
- SVN хранит дельты, git хранит снепшоты => git на порядки быстрее в работе
- если SVN сервер-центричный, то git распределнный, в нем основная работа идет локально
- Mercurial более простой и базовый

<img src="img/git_subversion.png" width=500>

#### Доменно-специфические расширения
- DVС
- Git LFS
- Git-Latex

# Логика работы

Main elements:
- Working directory / Worktree<br>рабочая директория
- Index / Staging area<br>текущий снепшот, включая изменения которые пойдут в комит
- Local repository<br>дерево снепшотов, хранящееся локально на машине арзработчкиа
- Remote repository<br>дереао снепшотов, хранящееся удаленно и доступное вовне
- Commit<br>фиксация изменений
- HEAD<br>pointer to the current commit (not necessary the latest one)

---

Репозиторий объектов хранит все возникаемые во время работы версии файлов (file_v1, file_v2, file_v3 ...). Это нужно, чтобы можно было быстро откатиться на любую версию в истории. Они добавляются, когда мы сообщаем, что планируем сохранять это изменение и вызываем команду git add

Если вручную удалили файл из WD - из индекса он никуда не девается (и соотвественно в хранилище объектов и в remote репоизтории)<br>Чтобы удалить файл из индекса, надо попросить git подхватить это изменение в индекс: для этого делаем git rm (поскольку физический файл уже удален, удаляется только из индекса) или git add (по смыслу применение удаления = удаление из индекса)<br>Если удалили случайно, можно просто откатить через git restore [file]

### добавление с индекс

__git add__
<br>adds changed and new files

__git add -u__
<br>adds only <u>tracked</u> changed files

Важно: команда git add добавляет в Staging Area не ссылку или дельту, а добавляет полноценную копию файла (снепшот). Это значит, что изменения после добавления не будут в комите

Все файлы, когда либо добавленные кодмандой git add и не удаленные оттуда переходят в статус __Tracked__

### проверка статуса

git status<br>выводит статусы изменений

### фиксация изменений

git commit<br>переводит содержимое Staging Area в Commit

git commit -m "commit description"<br>с добавлением комментария к коммиту

### синхронизация с remote (local -> remote)

__git push__<br>sends commit to the remote repository

__git push origin__<br>sends commit to the repository labeled as "origin"

### синхронизация с remote (local <- remote)

__git fetch__<br>from the remote 

__git merge__<br>

__git pull__<br>equivalent to git feetch + git merge

## Откат изменений<br>git reset / git restore / git rm

Коротко про эти три команды:
- git reset - двигает HEAD
- git restore - восстанавливает версию
- git rm - удаляет файл из репозитория, при следующем комите он будет исключен из индекса

### git reset



Если нужно откатить изменения, можно использовать команду<br>git reset [mode] [commit]

Параметры:
- [commit] задает куда откатываться
- [mode] задает, где именно откатывать
- [file] задает, что откатывать

git reset<br>removes files staged for commit

git reset --soft<br>устанавливает указатель HEAD на произвольный комит, больше ничего не меняет

git reset --mixed<br>устанавливает указатель HEAD на произвольный комит, Staging сбрасывает

git reset --hard<br>устанавливает указатель HEAD на произвольный комит и приводит версии всех файлов к его версии

По умолчанию включается режим mixed<br>
```git reset``` = `git reset --mixed`

git reset HEAD<br>откатывает к последнему комиту

git reset HEAD file1 file2<br>откатывает выборочно



| Команда                    | HEAD        | Staging Area     | Working Directory |
|----------------------------|-------------|-------------------|-------------------|
| `git reset --soft`         | ← двигает   | ❌ не трогает     | ❌ не трогает     |
| `git reset --mixed` (по умолчанию) | ← двигает   | ← сбрасывает     | ❌ не трогает     |
| `git reset --hard`         | ← двигает   | ← сбрасывает     | ← сбрасывает     |

При сбрасывании Staging Area: 
- если файл новый, он возвращается в статус "Untracked"
- если файл есть в индексе (ранее был хотя один комит с ним), то в статус "Not Staged for commit"

---

### git restore

Альтернатива - команда __git restore__. В отличие от git reset она не меняет state репозитория, она про локальные изменения<br>У нее два основных режима и они делают довольно разное

git restore file<br>возвращает файл к состоянию на последний комит (HEAD)

git restore --staged file<br>убирает копию файла из staging area, если он там есть (фактически undo для git add)

git restore --source=<commit><br>возвращает файл к состоянию на выбранный комит

| Команда                               | HEAD (ветка)        | Staging (Index)                           | Working Directory                         | Типичное назначение                                   |
|----------------------------------------|----------------------|--------------------------------------------|--------------------------------------------|--------------------------------------------------------|
| `git restore <file>`                   | ❌ не трогает        | ❌ не трогает                  | ✅ сбрасывает к HEAD             | Откатить изменения в WD                               |
| `git restore --staged <file>`          | ❌ не трогает        | ✅ сбрасывает                        | ❌ не трогает                               | Unstage файла                                         |





---

### git checkout
Используется для переключения Working Directory на выбраный комит. Можно использовать также для отката изменений (например если переключить на HEAD)

---

### git rm

__git rm__ удаляет файлы из репозитория

git rm<br>удаляет физически файл и  исключает из Staging Area

git rm --staged<br>исключает файл из Staging Area<br>то же самое, что `git restore --staged`?

git clean<br>удаляет все файлы с диска

| Команда                               | HEAD (ветка)        | Staging (Index)                           | Working Directory                         | Типичное назначение                                   |
|----------------------------------------|----------------------|--------------------------------------------|--------------------------------------------|--------------------------------------------------------|
| `git rm <file>`                        | ❌ не трогает        | ✅ удаляет запись файла из индекса         | ✅ удаляет файл с диска                    | Удалить файл из проекта                               |
| `git rm --cached <file>`               | ❌ не трогает        | ✅ удаляет из индекса                       | ❌ файл на диске остаётся                  | Перестать отслеживать файл                            |

### git mv
переименоваывпет файл или папку

### git stash

Часто возникает потрбеность переключиться на другую работу (например, поменять ветку или посмотреть какой-то из предыдущих комитов)

Команда __git stash__ - сохраняет текущий WD во временное хранилище. Временное хранилище - это стек, то есть можно несколько раз сохраянть<br>Используется когда для коммита текущих изменений недостаточно, но терять жалко

<img src="img/git_stash.png" width=500>

поместить WD в stash<br>
`git stash`

поместить WD в stash<br>
`git stash push`

добавить комментарий (если спустя время возращаеться)
<br>git stash save ""<br>

вывести все сохраненные WD
<br>git stash list<br>

вернуть и убрать из стеша
<br>git stash pop<br>

верунть но оставить в стеше
<br>git stash apply<br>

удалить из стеша (не пригодилось)
<br>git stash drop<br>

<br>включив untracked файлы
git stash show --include-untracked stash@{X}

Альтернативное решение - команда __git weorktree__

# Мониторинг
Команда __git reflog__ показывает все манипуляции с комитами: commit, pull, rebase, clone

git show

git status<br>команда показывает текущие статусы файлов в Working Directory:<br>
-untracked changes = есть в WD, но нет в индексе<br>
-changes not staged for commit = есть в индексе и есть изменения по сравнению с индексом<br>
-changes staged for commit = изменения, которые добавили через gir add

git log<br>показывает историю предыдущих комитов

git cat-file <sha><br>показывает содержимое объекта (файла, комита или каталога)

git ls-files<br>prints all files that are either in staging or in the index

### Навигация
**/*<br>
HEAD~1 = HEAD - 1

# Git Internals

В папке .git/objects хранятся все файлы, необходимые для работы репозитория<br>
Всего их три вида:
- __blob__<br>aka файлы с данными (не только текстовые)
- __commit__<br>ссылка на верхний каталог (tree), в котором есть измененные файлы + ссылка на предыдущий комит
- __tree__<br>каталог (список файлов)

<img src="img/git_objects1.png" width=400>

__Индекс__<br>
Текущий снепшот репозитория, содержит все когда либо добавленные и неуделнные файлы. Каждому файлу ставит в соотвествие хэш, по коотрому можно найти версию. При переключении между ветками индекс обновляется. При добавлении нового изменения обновляет хэш<br>
path -> (sha, metadata)

__История комитов__<br>История комитов = DAG-дерево, собранное из комитов (каждый комит содержит указатель на родительский или два родительских комита). Каждое новое изменение создает новый комит относительно текущего и тем самым продлоевает текущую ветку изменений. Каждый Merge объединяет изменения из двух комитов и объединяет ветки (поэтому DAG, а не дерево)

Важно, что git никак не хранит историю в виде лога событий, это просто текущий снепшот (HEAD), по которому эту историю можно восстановить.
Когда говорят, что история переписывается (например, при rebase), имеется в виду, что меняется дерево изменений - например, редактируются ссылки между комитами




# Branching

Branch = named reference to commit

<img src="img/git_branching3.png" width=300>

Работу с бранчами обычно визуализируют в виде некого timeline-а (см рисунок выше), но по сути это скорее дерево, из которого можно при необходимости отпачковать свою ветку. Её можно как угодно развивать, но поскольку вся польза в главной ветке (ствол), в конечном итоге любые ветки нужно синзронизировать с главной

Local branch = ref in local repository `master`<br>
Remote branch = ref in remote repository `origin/master`

<img src="img/git_branching2.png" width=450>

В каждый момент времени есть одна актуальная ветка, на нее указывает указатель __HEAD__. Ветка в свою очередь указывает на текущий комит в рамках этой ветки<br>Файл .git/refs/heads содержит актуальные комиты в каждой из веток<br>*Возможны ситуации, когда HEAD без ветки  указывает сразу на конкретный комит (detached HEAD)

## Просмотр веток
`git branch` выводит существующие ветки 

## Создание веток

git checkout -- file.txt<br>восстанавливает один файл из предыдущего коммита

git branch [branch_name]<br>альтернативная срециализированная команда, была доступна с первой версии

## Переключение между ветками

Ранее для переключения между комитами использовалась единая команда __git checkout__, но потом ее разделили на несколько более прозрачных: git branch / git_switch / git restore

Из-за этой неоднозначности, например, командой git checkout можно случайно переписать файл, если он совпадает с названием ветки<br>
`git checkout myfile.txt`

#### git checkout

git checkout branch<br>переключается на сосденюю ветку

git checkout branch<br>создает новую ветку

git checkout <commit_hash><br>переключается на конкретный комит

#### git switch

Отдельная команда появилась в 2019 году (вместе с git restore)

Переключение веток - безопасная команда, если в Working Directory есть конфликт с индексом требуемой версии, то команда выполняться не будет

При git switch
- берет снепшот нужной ветки (головной комит оттуда), по всем файлам оттуда проверяет, будет ли конфликт с текущей Working Directory
  - если есть конфликты (хэш файла из WD и хэш файла из снепшота нужной ветки не совпадают), то команда выполняться не будет
  - если нет конфликта (старые файлы не менялись или добавили только новые). Новые файлы останутся - и в WD, и в индексе (если их успели добавить комндой git add)

`git switch main`<br>переключение на нужную ветку

`git switch -c feature/login`<br>если сощдается новая ветка, можно сэкономить и вместо двух строчек (branch, switch) записать сразу одной

`git switch --discard-changes [branch]`<br>если в WD есть изменения в отслеживаемых файлах - забить на них, эти файлы откатятся на ветку [branch], а при последующем переключении обратно вернутся к последнему комиту

`git switch --merge [branch]`<br>если в WD есть изменения в отслеживаемых файлах - попытаться их перенести в целевую ветку. Если е при этом есть конфликты, они будут перенесены в новую версию файла - их надо будет вручную разрешать<br>это не merge, тут затрагивается только рабочая директория

## Объединение веток

### Merge
При объединении двух веток изменений A и B создается новая вресия AB => merge генерирует новый комит

Если изменения пересекаются (были изменены одни и те же файлы), возникает конфликт, который нужно решить. По умолчанию это делается руками


редкатор vim
как устоановить свой

<img src="img/git_merge1.png" width=500>

### Rebase
Rebase - альтернативный способ вмерджить feature-ветку в main-ветку. Вместо объединения изменений, делается параллельный перенос комитов. Как будто мы не параллельно разрабатвали, а сначала обновили основную ветку, а тут же применили все свои изменения разом

Важно, что при этом все сделанные ранее комиты заменяются на новые (то есть, как говорят в документации, переписывается история комитов)

Сценарий<br>Ход разработки feature-ветки может зависеть от main ветки (например, там баги фиксят). Чтобы фича в итоге завелась, нужно эти фиксы периодически подтяивать. Но многочисленные технические мерджи в feature ветке неифнормативны, повторяют комиты main-ветки и засоряют общее дерево, особенно когда много веток. Чтобы сделать дерево комитов почище, делают rebase

Почему не всегда использовать rebase?<br>Если пока вы писали вашу feture-ветку кто-то еще взял ее в работу, а вы потом её ребейзнули, все поломается

#### Interactive Rebase
Разрешение конфдиктов при стандартном rebase может быть утомительным, поэтому вместо того, чтобы накатывать комиты по одному на новую базу, можно это сделать, сконфигурировав все за один раз. Для этого создается конфигурационный файл со спсиком комитов по-порядку, который можно отредактировать. Для каждого комита нужно указать, что будем делать:
- pick - оставляем без изменений
- edit - отредкатировать комит
- split - разбить на немколькто комитов
- squash - оюхединить а одн комит
- delete - исключить комит
- reorder - посенять порядок комитов

### Sqush
Вместо последовательности небольших комитов можно их объединить в один комит

Сценарий<br>что-то забыли добавить в комит, и изменение его их нескольких

### Cherry-pick
Вместо вмердживания всей ветки может возникнуть необходимость повторить один конерткный комит из нее. Например, взять hotfix оттуда

<img src="img/git_cherrypick.png" width=500>

Оригинальный комит в этом никак не участвует, просто создается полностью новый комит с теми же изменениями. 
<br>Что такое те же изменения = git выделяет, какие именно файлы изменились в другой ветке по сравнению с общим комитом ($b \rightarrow r$) и пытается эти изменения перенести на текущую ветку. Если файлы пересекаются, будет конфликт, который надо разрешать руками

Команда выполняется из той ветки, где нужно применить изменение
`git cherry-pick <commit_hash>`

Можно сразу несколько комитов сделать
`git cherry-pick A B C`

В этом случае они применяются по-порядку (как при rebase)

Конфликты рарещаются стандартно, через исправление + `git add` и `git cherry-pick --continue`

Если хотим обновить WD, но без комита<br>
`git cherry-pick --no-commit <hash>`

Если хотим отредактировать сообщение в новом комите
`git cherry-pick -e <hash>`

По умолчанию берется из ориниганльного



# Синхронизация (git pull / git push)

Тут три основных команды:
- git push<br>переносит новые локальные комиты в remote репозиторий
- git fetch<br>загружает новые комиты из remote репозитория локально
- git pull<br>загружает новые комиты из remote репозитория локально и переключается на эту ветку

То есть<br>
git pull = git fetch + git merge / git rebase

<img src="img/git_remote1.png" width=500>

Синхронизация через git push и git pull может провоцировать конфликты: 
- с индексом - когда remote ветка и local ветка содержат разные изменения
- с Working Directory - когда решили обновить local ветку не зафиксировав текущие изменения
Их надо разруливать руками.

Как работает разрешегние конфликта<br>Обе версии конфликтующего файла сливаются в один файл, пересечения помечаются текстовыми маркерами. После корректировки нужно зафиксировать новую версию через git add и выполнить git merge

Способ объединения комитов слияние merge / упорядовичвание rebase задается

#### git push

Команда для пушинга изменений
`git push <remote> <remote_branch>`

По умолчанию пушится та ветка, на которой мы находимся (HEAD). Но можно явно указать<br>
`git push <remote> <local_branch>:<remote_branch>`

Запомнить выбор, чтобы потом не указывать, можно через опцию `-u`<br>
`git push origin -u branch`<br>

В этом случае далее можно писать просто `git push`

Также указать, с какой remote веткой будет синзронизироваться текущая ветка, можно прямо на уровне ветки через опцию set-upstream-to<br>
`git branch --set-upstream-to=branch <branch>`

А быстро посмотреть, как сконфигурировано по каждой ветке, так:<br>
`git branch -vv`



#### git fetch

Если локальный репозиторий ассоциирован с каким-то удаленным репозиторием (например, при git clone), в нем создаются ветки - локальные копии remote веток:<br>
origin/main<br>
origin/dev<br>
origin/feature

#### git pull

Загрузить обновления с remote репозитория<br>
`git pull origin`

Загрузить обновления с remote репозитория<br>
`git pull origin main`

Если есть конфликт, можно сделать rebase (прямо как в случае с синхронизацией веток в локльном репозитории). git rebase с remote не умеет работать, а pull может сделать
`git pull --rebase`





#  Конфигурация

Если не хочется опции указывать в командах, можно их запомнить и прописав в ini конфигурационных файлах

Скоуп действия опции:
- global - все репозитории (~/.gitconfig)
- repo - текущий репозиторий (.git/config)

Локальный конфиг находится в файле<br>
`.git/config`

Глобальный конфиг находится в файле<br>
`~/.gitconfig`

Типовая стурктура = блок, список key-value пар<br>
[core]<br>
editor = nano

Что может быть в станадртном файле конфигурации:
- параметры синхрнонизации всех когда-либо созданных веток<br>с какой remote веткой ассоциирована данная ветка
- зарегистрированные удаленные репозитории<br>блоки с именем branch
- режимы по умолчанию при командах git pull<br>

<img src="img/git_config.png" width=500>

Установить редактор по умолчанию для всех репозиториев в системе<br>
`git config --global core.editor "nano"`

Установить редактор по умолчанию для данного репозитория, в котором мы находимся<br>
`git config core.editor "nano"`

### gitignore / gitkeep

__.gitignore__
Файл для автоматического исключения определенных типов файлов из трекинга. Помещается в корень либо подпапки (тогда действует только на нее)

пример
**/*
.DS_Store
    
__.gitkeep__
Файл 

# Аутентификация

Authentication methods:
- __HTTPS__<br>tokens are sent to server via HTTP protocol (previously passwords, but now are prohibited)
- __SSH__<br>connection using generated key pair

To check which one is used:<br>```git remote -v```<br>and see the prefix of the link

Credentials<br>
login = github login
password = recently generated token (if not saved then lost)

To store them:
- in file<br>```git config --global credential.helper store```
- in keychain<br> ```git config --global credential.helper osxkeychain```
- in memory<br> ```git config --global credential.helper 'cache --timeout=3600'```

```git remote``` - shows the list of linked remote repositories
```git remote -v``` - adds more details (separate for push and fetch)
```git remote add``` - shows the list of linked remote repositories

### Настройка SSH аутентификации

```python

# генерируем пару ключей
ssh-keygen -t ed25519

# можно RSA но устраавевее
ssh-keygen -t rsa

Добавляем на сервисе: GitHub → Settings → SSH and GPG keys

# указываем, что к Remote теперь будем подключаться по SSH
git remote set-url origin git@github.com:konstantin-kotochigov/explainy.git

# если при создании прописал имя ключа, его надо руками класть в хранилизе ключей .ssh
mv github.key ~/.ssh/
chmod 600 ~/.ssh/github.key 
mv github.key.pub ~/.ssh/
chmod 644 ~/.ssh/github.key.pub

# добавляем в keychain
ssh-add --apple-use-keychain ~/.ssh/github.key

# прописать, какой ключ используем для хоста в .ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile ~/.ssh/id_ed25519
  AddKeysToAgent yes
  UseKeychain yes


# проверка
ssh-add -l

# проверка связи
ssh -T git@github.com

```

### Git Hooks

You can define a number of callbacks (hooks) that gonna be run whenever some trigger is fired

Possible triggers:
- pre-commit
- post-commit
- pre-push
- post-push

Where do the hooks reside:
- .git - internal git files
- .git/hooks - callback scripts

There is a number of samples already prepacked.

To enable a hook just put a script with corresponding name (or rename an example).

**NOTE** This directory is local, it won't go to global repo.

Scripts use arguments sys.argv[1] and return values (non-zero => the process stops)

Scripts could be:
  - bash
  - python
  - go

<img src="img/dev_process.png" width=750>

Use-cases
- beautify the code
- check file names for ASCII compatibility
- enhance commit message
- enforce detailed commit messages
- run test build
        
        it would help always maintain code in buildable state
        not sure it's relevant for Python since you don't compile
- run unit tests

It seems that they don pretty much the same as CI/CD tools like Jenkins.

You can check the list of files being updated in curent commit by running:
```
git-diff --name-only 
```

## git-diff

It's a file comparison utility. It can compare different types of files

Сравнивает WD с текущим комитом (HEAD): зеленое - то что предлагает текущее изменение<br>
git-diff

Сравнивает WD с произвольным комитом<br>
git-diff HEAD~1

Сравнивает только конекретный файл<br>
git-diff HEAD file

Сравнивает 2 произвольных комита<br>
git-diff HEAD HEAD~1

Сравнивает два конкретных файла<br>
git-diff file1 file2

Сравнивает две ветки<br>
git-diff 

### git-stash

Temporarilty stores current working changes WITHOUT commiting them.

# Git расширения

## GitHub Actions

`GitHub Actions` = инструмент автоматизации процесса разработки, глубоко интегриированный с GitHub

Он позволяет реализовать event-driven стратегию работы - срабатвет триггер => выполняется дейтсвие. 

Когда репозиторий большой, требуется большое кол-во менеджмента, как техничсеокго (прогнать CI/CD пайплайн для каждого Pull-Request) так и административного (поприветствовать нового конрибьютора, написать статус в комментарий, нащначить на кого-то тикет)

В UI есть вкладка Actions с маркетплейсом преднастроенных шаблонов и примерами конфиг файлов

В Github Actions автоматизация = конфиг файл формата YAML, который добавляется в репозиторий и в котором содержатся все триггеры и едействия на них

### Синтаксис конфига

Для выполнения команд Github временно выделяет свою dedicated средк<br>
`runs_on` - на какой ОС выполнять

`on` - перечисление триггеров в формате событие / условие

`jobs` - список джобов, которые будет выполняться<br>по умолчанию параллельно, но можно указать `depends_on`

`steps` - последовательность шагов в каждом джобе, которая будет выполняться

`uses` - преднастроенное дейтсвие из репозитория Actions<br>
`run` - запустить свою bash команду<br>
`with` - передаваемые параметры

Можно испольщовать secret переменные<br>
`${{secrets.username}} и ${{secrets.passwird}}`



### Gitlab - CI/CD

### BitBucket